<style>
.retailer-hero { padding: 34px 36px; border-radius: 18px; color: #111827; background: linear-gradient(135deg, #f8fafc 0%, #ecfeff 42%, #fff7ed 100%); border: 1px solid #d1d5db; box-shadow: 0 18px 48px rgba(15, 23, 42, 0.12); }
.retailer-hero h1 { font-size: 44px; line-height: 1.02; margin: 0 0 12px 0; letter-spacing: 0; }
.retailer-hero p { max-width: 920px; font-size: 17px; line-height: 1.55; margin: 8px 0 0 0; color: #374151; }
.retailer-badges { display: flex; flex-wrap: wrap; gap: 10px; margin-top: 22px; }
.retailer-badge { padding: 9px 12px; border-radius: 999px; background: rgba(255, 255, 255, 0.82); border: 1px solid #cbd5e1; font-weight: 700; font-size: 13px; }
.retailer-note { margin-top: 22px; padding: 13px 16px; border-left: 5px solid #0f766e; background: rgba(240, 253, 250, 0.85); border-radius: 10px; color: #134e4a; }
</style>
<section class='retailer-hero'>
  <h1>Retailer IQ Live Demo</h1>
  <p><strong>RecommenderIQ</strong> is framed as a decoupled recommendation system managed by a recommendation-agent capability hosted by <code>search-enrichment-agent</code>. This notebook calls the live <code>holidaypeakhub405</code> dev services, visualizes what already exists, and shows which signals can be plugged into a retailer-grade recommender system.</p>
  <div class='retailer-badges'>
    <span class='retailer-badge'>Environment: holidaypeakhub405 dev</span>
    <span class='retailer-badge'>Gateway: Azure API Management</span>
    <span class='retailer-badge'>Runtime: REST + MCP</span>
    <span class='retailer-badge'>Model path: classical ML first</span>
    <span class='retailer-badge'>Agent role: orchestrate, govern, explain</span>
  </div>
  <div class='retailer-note'>This deck is executable. Run the cells top to bottom before presenting; if APIM requires a subscription key, set <code>APIM_SUBSCRIPTION_KEY</code> in the notebook kernel environment.</div>
</section>

## What This Notebook Proves

1. The deployed platform already has the domains needed for a full retailer recommendation system: Product IQ, Customer IQ, Search/Intent IQ, Commerce IQ, Inventory IQ, Logistics IQ, and governance/HITL.
2. The new <code>recommendation-agent</code> capability is intentionally smaller than Retailer IQ. It manages recommendation candidate generation, ranking, composition, feedback, explanation, and model status.
3. The hot path should stay deterministic/classical ML first. The agent can invoke Foundry for intent, explanation, evaluation, or complex orchestration, but the model artifact is trained and tracked outside the request path.
4. Browser readiness is a contract: recommendation cards are only customer-facing when model, feature, policy, inventory, freshness, and explanation gates are satisfied.

In [ ]:
from __future__ import annotations

import html
import json
import os
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib import error, parse, request

from IPython.display import HTML, Markdown, display

DEFAULT_ENV_NAME = 'holidaypeakhub405'
DEFAULT_APIM_BASE_URL = 'https://holidaypeakhub405-dev-apim.azure-api.net'
DEFAULT_UI_URL = 'https://blue-meadow-00fcb8810.4.azurestaticapps.net'

ENV_NAME = os.getenv('AZURE_ENV_NAME') or os.getenv('AZD_ENV_NAME') or DEFAULT_ENV_NAME
RESOURCE_GROUP = os.getenv('AZURE_RESOURCE_GROUP') or f'{ENV_NAME}-dev-rg'
REQUEST_TIMEOUT_SECONDS = float(os.getenv('HOLIDAY_PEAK_TIMEOUT_SECONDS', '8'))
APIM_SUBSCRIPTION_KEY = (
    os.getenv('APIM_SUBSCRIPTION_KEY')
    or os.getenv('OCP_APIM_SUBSCRIPTION_KEY')
    or os.getenv('AZURE_APIM_SUBSCRIPTION_KEY')
    or ''
)

def _repo_root() -> Path:
    current = Path.cwd()
    if (current / 'azure.yaml').exists():
        return current
    for parent in current.parents:
        if (parent / 'azure.yaml').exists():
            return parent
    return current

ROOT = _repo_root()

def _normalize_url(value: str | None) -> str:
    if not value:
        return ''
    return value.strip().rstrip('/')

def _run_json_command(args: list[str], timeout_seconds: float = 15) -> Any | None:
    try:
        output = subprocess.check_output(
            args, stderr=subprocess.DEVNULL, text=True, timeout=timeout_seconds
        )
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        return None
    try:
        return json.loads(output)
    except json.JSONDecodeError:
        return None

def _first_string(values: Any) -> str:
    if isinstance(values, list):
        for value in values:
            if isinstance(value, str) and value:
                return value
            if isinstance(value, dict):
                for key in ('gatewayUrl', 'defaultHostname'):
                    candidate = value.get(key)
                    if isinstance(candidate, str) and candidate:
                        return candidate
    if isinstance(values, str):
        return values
    return ''

def discover_apim_base_url() -> str:
    configured = _normalize_url(
        os.getenv('HOLIDAY_PEAK_APIM_BASE_URL')
        or os.getenv('NEXT_PUBLIC_API_BASE_URL')
        or os.getenv('BASE_URL')
    )
    if configured:
        return configured

    discovered = _run_json_command([
        'az', 'apim', 'list',
        '--resource-group', RESOURCE_GROUP,
        '--query', '[].gatewayUrl',
        '-o', 'json',
    ])
    return _normalize_url(_first_string(discovered)) or DEFAULT_APIM_BASE_URL

def discover_ui_url() -> str:
    configured = _normalize_url(os.getenv('HOLIDAY_PEAK_UI_URL') or os.getenv('NEXT_PUBLIC_UI_URL'))
    if configured:
        return configured
    discovered = _run_json_command([
        'az', 'staticwebapp', 'list',
        '--resource-group', RESOURCE_GROUP,
        '--query', '[].defaultHostname',
        '-o', 'json',
    ])
    hostname = _first_string(discovered)
    return _normalize_url(f'https://{hostname}' if hostname and not hostname.startswith('http') else hostname) or DEFAULT_UI_URL

APIM_BASE_URL = discover_apim_base_url()
UI_URL = discover_ui_url()

def request_headers(url: str) -> dict[str, str]:
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json',
        'User-Agent': 'holiday-peak-retailer-iq-notebook',
    }
    if APIM_SUBSCRIPTION_KEY and url.startswith(APIM_BASE_URL):
        headers['Ocp-Apim-Subscription-Key'] = APIM_SUBSCRIPTION_KEY
    return headers

display(HTML(f'''
<style>
.config-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 14px; margin: 18px 0; }}
.config-card {{ border: 1px solid #d1d5db; border-radius: 12px; padding: 14px 16px; background: #ffffff; box-shadow: 0 6px 20px rgba(15, 23, 42, 0.07); }}
.config-card .label {{ color: #64748b; font-size: 12px; text-transform: uppercase; font-weight: 800; letter-spacing: 0.04em; }}
.config-card .value {{ color: #0f172a; font-size: 15px; font-weight: 800; margin-top: 6px; overflow-wrap: anywhere; }}
.config-card .hint {{ color: #64748b; font-size: 12px; margin-top: 8px; }}
</style>
<div class='config-grid'>
  <div class='config-card'><div class='label'>Environment</div><div class='value'>{html.escape(ENV_NAME)}</div><div class='hint'>Resource group: {html.escape(RESOURCE_GROUP)}</div></div>
  <div class='config-card'><div class='label'>APIM Gateway</div><div class='value'>{html.escape(APIM_BASE_URL)}</div><div class='hint'>Override with HOLIDAY_PEAK_APIM_BASE_URL or NEXT_PUBLIC_API_BASE_URL.</div></div>
  <div class='config-card'><div class='label'>UI Host</div><div class='value'>{html.escape(UI_URL)}</div><div class='hint'>Override with HOLIDAY_PEAK_UI_URL.</div></div>
  <div class='config-card'><div class='label'>APIM Subscription Key</div><div class='value'>{'configured' if APIM_SUBSCRIPTION_KEY else 'not configured'}</div><div class='hint'>Set APIM_SUBSCRIPTION_KEY only in the kernel environment.</div></div>
</div>
'''))

In [ ]:
AGENT_SERVICES: list[dict[str, str]] = [
    {'domain': 'CRM IQ', 'service': 'crm-profile-aggregation', 'layer': 'Customer IQ', 'signal': 'Unified profile, identifiers, consent-aware customer context'},
    {'domain': 'CRM IQ', 'service': 'crm-segmentation-personalization', 'layer': 'Customer IQ', 'signal': 'Segments, propensities, personalization context'},
    {'domain': 'CRM IQ', 'service': 'crm-campaign-intelligence', 'layer': 'Customer IQ', 'signal': 'Offer affinity, channel response, campaign timing'},
    {'domain': 'CRM IQ', 'service': 'crm-support-assistance', 'layer': 'Customer IQ', 'signal': 'Support sentiment, friction, service recovery context'},
    {'domain': 'E-Commerce IQ', 'service': 'ecommerce-catalog-search', 'layer': 'Intent IQ', 'signal': 'Search intent and retrieval context consumed by browsing flows'},
    {'domain': 'E-Commerce IQ', 'service': 'ecommerce-product-detail-enrichment', 'layer': 'Product Experience IQ', 'signal': 'PDP enrichment, attributes, related content'},
    {'domain': 'E-Commerce IQ', 'service': 'ecommerce-cart-intelligence', 'layer': 'Commerce IQ', 'signal': 'Cart co-occurrence, bundle, and abandonment signals'},
    {'domain': 'E-Commerce IQ', 'service': 'ecommerce-checkout-support', 'layer': 'Commerce IQ', 'signal': 'Pricing, checkout friction, shipping decision context'},
    {'domain': 'E-Commerce IQ', 'service': 'ecommerce-order-status', 'layer': 'Commerce IQ', 'signal': 'Post-purchase state and service expectations'},
    {'domain': 'Inventory IQ', 'service': 'inventory-health-check', 'layer': 'Supply IQ', 'signal': 'Stock health, risk bands, sellable availability'},
    {'domain': 'Inventory IQ', 'service': 'inventory-jit-replenishment', 'layer': 'Supply IQ', 'signal': 'Demand velocity, replenishment timing, scarcity signals'},
    {'domain': 'Inventory IQ', 'service': 'inventory-reservation-validation', 'layer': 'Supply IQ', 'signal': 'Reservation feasibility and allocation confidence'},
    {'domain': 'Inventory IQ', 'service': 'inventory-alerts-triggers', 'layer': 'Supply IQ', 'signal': 'Exceptions, outages, threshold events'},
    {'domain': 'Logistics IQ', 'service': 'logistics-carrier-selection', 'layer': 'Promise IQ', 'signal': 'Carrier speed, cost, and service-level trade-offs'},
    {'domain': 'Logistics IQ', 'service': 'logistics-eta-computation', 'layer': 'Promise IQ', 'signal': 'ETA confidence and delivery promise windows'},
    {'domain': 'Logistics IQ', 'service': 'logistics-returns-support', 'layer': 'Promise IQ', 'signal': 'Return risk, reverse logistics, recovery flow'},
    {'domain': 'Logistics IQ', 'service': 'logistics-route-issue-detection', 'layer': 'Promise IQ', 'signal': 'Route disruption and delay risk'},
    {'domain': 'Product IQ', 'service': 'product-management-normalization-classification', 'layer': 'Product IQ', 'signal': 'Taxonomy, normalized attributes, product feature hygiene'},
    {'domain': 'Product IQ', 'service': 'product-management-acp-transformation', 'layer': 'Product IQ', 'signal': 'ACP-ready catalog payloads and channel formatting'},
    {'domain': 'Product IQ', 'service': 'product-management-consistency-validation', 'layer': 'Product IQ', 'signal': 'Completeness, trust, and quality validation'},
    {'domain': 'Product IQ', 'service': 'product-management-assortment-optimization', 'layer': 'Product IQ', 'signal': 'Category mix, assortment coverage, portfolio constraints'},
    {'domain': 'Search and Recommendation IQ', 'service': 'search-enrichment-agent', 'layer': 'RecommenderIQ', 'signal': 'Correlated products, recommendation-agent candidate/rank/compose/explain/feedback'},
    {'domain': 'Product Truth IQ', 'service': 'truth-ingestion', 'layer': 'Product IQ', 'signal': 'Source ingestion and product event intake'},
    {'domain': 'Product Truth IQ', 'service': 'truth-enrichment', 'layer': 'Product IQ', 'signal': 'Approved product truth, feature extraction, enrichment state'},
    {'domain': 'Product Truth IQ', 'service': 'truth-hitl', 'layer': 'Governance IQ', 'signal': 'Human-in-the-loop review and policy gates'},
    {'domain': 'Product Truth IQ', 'service': 'truth-export', 'layer': 'Governance IQ', 'signal': 'Publication readiness and channel export state'},
]

CORE_SERVICES = [
    {'domain': 'Core Platform', 'service': 'crud', 'layer': 'Transactional Core', 'signal': 'Products, users, carts, orders, inventory, reviews via /api'},
    {'domain': 'Experience', 'service': 'ui', 'layer': 'Customer Experience', 'signal': 'Browsing surface that consumes ready recommendation cards'},
]

DOMAIN_ORDER = ['CRM IQ', 'E-Commerce IQ', 'Product IQ', 'Product Truth IQ', 'Inventory IQ', 'Logistics IQ', 'Search and Recommendation IQ']

def render_service_inventory(services: list[dict[str, str]]) -> None:
    grouped: dict[str, list[dict[str, str]]] = {domain: [] for domain in DOMAIN_ORDER}
    for item in services:
        grouped.setdefault(item['domain'], []).append(item)

    sections = []
    for domain, items in grouped.items():
        if not items:
            continue
        cards = ''.join(
            f"<div class='service-card'><div class='service-name'>{html.escape(item['service'])}</div>"
            f"<div class='service-layer'>{html.escape(item['layer'])}</div>"
            f"<div class='service-signal'>{html.escape(item['signal'])}</div></div>"
            for item in items
        )
        sections.append(
            f"<section class='domain-band'><div class='domain-title'>{html.escape(domain)}"
            f"<span>{len(items)} services</span></div><div class='service-grid'>{cards}</div></section>"
        )

    display(HTML(f'''
<style>
.inventory-wrap {{ margin: 18px 0 6px; }}
.domain-band {{ margin: 14px 0; padding: 16px; border-radius: 14px; border: 1px solid #d7dde6; background: #ffffff; }}
.domain-title {{ display: flex; justify-content: space-between; gap: 16px; align-items: center; font-size: 18px; font-weight: 900; color: #111827; margin-bottom: 12px; }}
.domain-title span {{ font-size: 12px; color: #475569; background: #f1f5f9; border-radius: 999px; padding: 5px 9px; }}
.service-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 10px; }}
.service-card {{ min-height: 118px; border-radius: 10px; padding: 12px; background: linear-gradient(180deg, #ffffff 0%, #f8fafc 100%); border: 1px solid #e2e8f0; }}
.service-name {{ font-weight: 850; color: #0f172a; font-size: 14px; overflow-wrap: anywhere; }}
.service-layer {{ color: #0369a1; font-size: 12px; font-weight: 800; margin-top: 6px; }}
.service-signal {{ color: #475569; font-size: 12px; line-height: 1.45; margin-top: 7px; }}
</style>
<div class='inventory-wrap'>
  <h2>Existing Service Map: {len(services)} agents + CRUD + UI</h2>
  <p style='color:#475569'>Each health call in the next section targets these deployed service boundaries. The recommendation system consumes their signals through contracts, not by owning their data.</p>
  {''.join(sections)}
</div>
'''))

render_service_inventory(AGENT_SERVICES)

In [ ]:
def render_retailer_iq_map() -> None:
    flows = [
        ('Product IQ', 'Truth services, product-management agents, approved attributes'),
        ('Customer IQ', 'CRM agents, consent-aware profile and segment context'),
        ('Intent IQ', 'Search, PDP, cart, checkout, order context'),
        ('Supply and Promise IQ', 'Inventory availability, ETA, carrier, returns, route risk'),
        ('RecommenderIQ', 'Candidate generation, ranking, card composition, explanation, feedback'),
        ('Customer Experience', 'UI consumes only ready, governed recommendation cards'),
    ]
    flow_html = ''.join(
        f"<div class='iq-node'><div class='iq-title'>{html.escape(title)}</div><div>{html.escape(body)}</div></div>"
        for title, body in flows
    )
    hot_path = [
        ('1', 'Collect signals', 'REST, MCP, Event Hubs, and approved projections'),
        ('2', 'Generate candidates', 'Explicit products plus correlated products from search enrichment'),
        ('3', 'Rank', 'Deterministic/classical ML model with policy and context features'),
        ('4', 'Compose cards', 'UI-ready payloads with reason codes and evidence'),
        ('5', 'Capture feedback', 'Impression, click, add-to-cart, purchase, dismissal'),
    ]
    hot_html = ''.join(
        f"<div class='step'><span>{n}</span><strong>{html.escape(title)}</strong><p>{html.escape(body)}</p></div>"
        for n, title, body in hot_path
    )

    display(HTML(f'''
<style>
.iq-shell {{ margin: 20px 0; padding: 20px; border-radius: 18px; background: #0f172a; color: #f8fafc; }}
.iq-shell h2 {{ color: #ffffff; margin: 0 0 8px; }}
.iq-shell .sub {{ color: #cbd5e1; margin: 0 0 18px; }}
.iq-flow {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(210px, 1fr)); gap: 12px; }}
.iq-node {{ border: 1px solid rgba(255,255,255,0.18); background: rgba(255,255,255,0.08); border-radius: 12px; padding: 14px; min-height: 126px; color: #e5e7eb; }}
.iq-title {{ color: #67e8f9; font-weight: 900; margin-bottom: 8px; }}
.model-row {{ display: grid; grid-template-columns: 1fr 1fr; gap: 14px; margin-top: 18px; }}
.model-card {{ background: #ffffff; color: #111827; border-radius: 12px; padding: 14px; }}
.model-card h3 {{ margin: 0 0 8px; font-size: 16px; }}
.model-card p {{ color: #475569; margin: 0; line-height: 1.45; }}
.hot-path {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 10px; margin-top: 14px; }}
.step {{ background: #fff7ed; color: #111827; border: 1px solid #fed7aa; border-radius: 10px; padding: 12px; }}
.step span {{ display: inline-grid; place-items: center; width: 26px; height: 26px; border-radius: 999px; background: #0f766e; color: white; font-weight: 900; margin-bottom: 8px; }}
.step strong {{ display: block; font-size: 14px; }}
.step p {{ margin: 6px 0 0; font-size: 12px; color: #475569; }}
@media (max-width: 820px) {{ .model-row {{ grid-template-columns: 1fr; }} }}
</style>
<section class='iq-shell'>
  <h2>Retailer IQ as a Separate System</h2>
  <p class='sub'>Retailer IQ is the decisioning layer. RecommenderIQ is one capability inside it, and recommendation-agent is the agent-managed surface for that capability.</p>
  <div class='iq-flow'>{flow_html}</div>
  <div class='model-row'>
    <div class='model-card'><h3>Agent role</h3><p>The agent orchestrates contracts, explains decisions, records feedback, reports model status, and escalates complex reasoning through Foundry. It is not the model artifact.</p></div>
    <div class='model-card'><h3>Model lifecycle</h3><p>Training and tracking stay offline using MLET-style ML Canvas, MLflow, data lineage, registry stages, model cards, and explicit promotion gates.</p></div>
  </div>
  <div class='hot-path'>{hot_html}</div>
</section>
'''))

render_retailer_iq_map()

In [ ]:
@dataclass(frozen=True)
class HttpCallResult:
    label: str
    method: str
    url: str
    status_code: int | None
    ok: bool
    latency_ms: float
    body: Any
    message: str

def _json_or_text(raw: str) -> Any:
    if not raw:
        return {}
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'raw': raw[:1600]}

def call_http(
    method: str,
    path_or_url: str,
    *,
    payload: dict[str, Any] | None = None,
    label: str | None = None,
    timeout_seconds: float = REQUEST_TIMEOUT_SECONDS,
) -> HttpCallResult:
    url = path_or_url if path_or_url.startswith('http') else f'{APIM_BASE_URL}{path_or_url}'
    data = json.dumps(payload).encode('utf-8') if payload is not None else None
    req = request.Request(url, data=data, headers=request_headers(url), method=method.upper())
    started = time.perf_counter()
    try:
        with request.urlopen(req, timeout=timeout_seconds) as response:
            raw = response.read(6000).decode('utf-8', errors='replace')
            latency_ms = (time.perf_counter() - started) * 1000
            status_code = int(getattr(response, 'status', 0) or response.getcode())
            return HttpCallResult(label or url, method.upper(), url, status_code, 200 <= status_code < 300, latency_ms, _json_or_text(raw), 'ok')
    except error.HTTPError as exc:
        raw = exc.read(6000).decode('utf-8', errors='replace')
        latency_ms = (time.perf_counter() - started) * 1000
        return HttpCallResult(label or url, method.upper(), url, exc.code, False, latency_ms, _json_or_text(raw), str(exc))
    except error.URLError as exc:
        latency_ms = (time.perf_counter() - started) * 1000
        return HttpCallResult(label or url, method.upper(), url, None, False, latency_ms, {}, str(exc.reason))
    except TimeoutError as exc:
        latency_ms = (time.perf_counter() - started) * 1000
        return HttpCallResult(label or url, method.upper(), url, None, False, latency_ms, {}, str(exc))

def classify_call(result: HttpCallResult) -> tuple[str, str]:
    body_text = json.dumps(result.body, default=str).lower()[:1200]
    message = result.message.lower()
    if result.ok:
        return 'ready', 'Ready'
    if result.status_code in (401, 403):
        return 'auth', 'Needs APIM key or auth'
    if result.status_code and result.status_code >= 500:
        return 'backend', 'Backend or upstream issue'
    if 'no healthy upstream' in body_text or 'no healthy upstream' in message:
        return 'backend', 'No healthy upstream'
    if result.status_code == 404:
        return 'route', 'Route not published'
    return 'offline', 'Unavailable'

HEALTH_TARGETS = [
    {**service, 'kind': 'agent', 'path': f"/agents/{service['service']}/health"}
    for service in AGENT_SERVICES
] + [
    {'domain': 'Core Platform', 'service': 'crud', 'layer': 'Transactional Core', 'signal': 'CRUD API health', 'kind': 'crud', 'path': '/api/health'},
    {'domain': 'Experience', 'service': 'ui', 'layer': 'Customer Experience', 'signal': 'Static Web App shell', 'kind': 'ui', 'path': UI_URL},
]

def run_health_sweep(max_workers: int = 10) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(call_http, 'GET', target['path'], label=target['service']): target
            for target in HEALTH_TARGETS
        }
        for future in as_completed(futures):
            target = futures[future]
            result = future.result()
            status_key, status_label = classify_call(result)
            rows.append({
                **target,
                'status_key': status_key,
                'status_label': status_label,
                'status_code': result.status_code,
                'ok': result.ok,
                'latency_ms': round(result.latency_ms, 1),
                'message': result.message,
                'body': result.body,
                'url': result.url,
            })
    return sorted(rows, key=lambda row: (row['domain'], row['service']))

def render_health_sweep(rows: list[dict[str, Any]]) -> None:
    palette = {
        'ready': ('#ecfdf5', '#047857', '#34d399'),
        'auth': ('#eff6ff', '#1d4ed8', '#93c5fd'),
        'backend': ('#fff7ed', '#c2410c', '#fdba74'),
        'route': ('#fef2f2', '#b91c1c', '#fca5a5'),
        'offline': ('#f8fafc', '#475569', '#cbd5e1'),
    }
    counts: dict[str, int] = {}
    for row in rows:
        counts[row['status_key']] = counts.get(row['status_key'], 0) + 1
    summary = ''.join(
        f"<div class='metric'><div class='metric-value'>{counts.get(key, 0)}</div><div>{label}</div></div>"
        for key, label in [('ready', 'ready'), ('auth', 'auth gated'), ('backend', 'backend issue'), ('route', 'route gap'), ('offline', 'offline')]
    )
    cards = []
    for row in rows:
        bg, fg, border = palette[row['status_key']]
        code = 'n/a' if row['status_code'] is None else str(row['status_code'])
        cards.append(
            f"<div class='health-card' style='background:{bg}; border-color:{border}'>"
            f"<div class='health-top'><strong>{html.escape(row['service'])}</strong><span style='color:{fg}'>{html.escape(row['status_label'])}</span></div>"
            f"<div class='health-meta'>{html.escape(row['domain'])} | HTTP {code} | {row['latency_ms']} ms</div>"
            f"<div class='health-msg'>{html.escape(str(row['message'])[:140])}</div>"
            f"</div>"
        )
    display(HTML(f'''
<style>
.health-summary {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 10px; margin: 16px 0; }}
.metric {{ border-radius: 12px; padding: 12px; background: #ffffff; border: 1px solid #e2e8f0; text-align: center; }}
.metric-value {{ font-size: 28px; font-weight: 950; color: #0f172a; }}
.health-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(270px, 1fr)); gap: 10px; }}
.health-card {{ border: 1px solid; border-radius: 11px; padding: 12px; min-height: 104px; }}
.health-top {{ display: flex; justify-content: space-between; gap: 12px; align-items: flex-start; }}
.health-top strong {{ overflow-wrap: anywhere; color: #111827; }}
.health-top span {{ font-size: 12px; font-weight: 900; white-space: nowrap; }}
.health-meta {{ color: #64748b; font-size: 12px; margin-top: 8px; }}
.health-msg {{ color: #334155; font-size: 12px; margin-top: 7px; overflow-wrap: anywhere; }}
</style>
<h2>Live Service Health Sweep</h2>
<p style='color:#475569'>Timestamp: {datetime.now(timezone.utc).isoformat()} | Gateway: {html.escape(APIM_BASE_URL)}</p>
<div class='health-summary'>{summary}</div>
<div class='health-grid'>{''.join(cards)}</div>
'''))

health_results = run_health_sweep()
render_health_sweep(health_results)

In [ ]:
from collections import Counter

health_counts = Counter(row['status_key'] for row in health_results)
health_summary_rows = ''.join(
    f"<tr><td><strong>{html.escape(status)}</strong></td><td>{count}</td></tr>"
    for status, count in sorted(health_counts.items())
)
display(HTML(f'''
<h3>Health Sweep Summary</h3>
<table style="border-collapse:collapse; min-width:320px;">
  <thead><tr><th style="text-align:left; padding:8px; background:#0f172a; color:white;">Status</th><th style="text-align:left; padding:8px; background:#0f172a; color:white;">Count</th></tr></thead>
  <tbody>{health_summary_rows}</tbody>
</table>
'''))

In [ ]:
CONTRIBUTION_ROWS = [
    ('Product IQ', 'truth-ingestion, truth-enrichment, product-management-*', 'Approved product graph, taxonomy, normalized attributes, completeness, assortment context', 'Product Truth Layer and product-management services', 'Product feature projection is fresh and approved'),
    ('Customer IQ', 'crm-profile-aggregation, crm-segmentation-personalization, crm-campaign-intelligence, crm-support-assistance', 'Consent-aware customer context, segments, offer affinity, support friction', 'CRM bounded contexts', 'Customer feature projection is consented, scoped, and fresh'),
    ('Intent IQ', 'ecommerce-catalog-search, ecommerce-product-detail-enrichment, ecommerce-cart-intelligence', 'Query intent, PDP context, cart co-occurrence, bundle hints', 'E-commerce agents and UI session state', 'Session context has a stable anonymous or known subject ref'),
    ('Supply IQ', 'inventory-health-check, inventory-jit-replenishment, inventory-reservation-validation, inventory-alerts-triggers', 'Availability, stock risk, replenishment velocity, allocation confidence', 'Inventory agents and CRUD inventory routes', 'Sellable inventory is known for the candidate SKU'),
    ('Promise IQ', 'logistics-carrier-selection, logistics-eta-computation, logistics-returns-support, logistics-route-issue-detection', 'Delivery promise, return risk, route disruption, carrier trade-offs', 'Logistics agents', 'Promise window is available for the customer region'),
    ('RecommenderIQ', 'search-enrichment-agent / recommendation-agent', 'Candidates, rank, compose, explain, feedback, model status', 'Recommendation-agent capability', 'Model, feature, policy, evidence, and UI-readiness gates pass'),
]

def render_contribution_matrix(rows: list[tuple[str, str, str, str, str]]) -> None:
    body = ''.join(
        '<tr>'
        f"<td><strong>{html.escape(layer)}</strong></td>"
        f"<td><code>{html.escape(services)}</code></td>"
        f"<td>{html.escape(signal)}</td>"
        f"<td>{html.escape(owner)}</td>"
        f"<td>{html.escape(gate)}</td>"
        '</tr>'
        for layer, services, signal, owner, gate in rows
    )
    display(HTML(f'''
<style>
.matrix {{ width: 100%; border-collapse: separate; border-spacing: 0; border: 1px solid #dbe3ef; border-radius: 14px; overflow: hidden; }}
.matrix th {{ background: #0f172a; color: #ffffff; text-align: left; padding: 11px; font-size: 13px; }}
.matrix td {{ padding: 11px; border-top: 1px solid #e5e7eb; vertical-align: top; color: #334155; font-size: 13px; line-height: 1.45; }}
.matrix tr:nth-child(even) td {{ background: #f8fafc; }}
.matrix code {{ white-space: normal; color: #0f766e; font-weight: 800; }}
</style>
<h2>What Can Already Feed the Retailer's Recommender System</h2>
<table class='matrix'>
  <thead><tr><th>IQ layer</th><th>Existing services</th><th>Recommendation contribution</th><th>State owner</th><th>Ready when</th></tr></thead>
  <tbody>{body}</tbody>
</table>
'''))

render_contribution_matrix(CONTRIBUTION_ROWS)

In [ ]:
CANDIDATE_REQUEST = {
    'tenant_id': ENV_NAME,
    'customer_ref': 'loyalty:demo-gold-001',
    'session_id': 'presentation-session-001',
    'intent': 'premium wireless headphones for business travel',
    'query': 'noise cancelling headphones with travel case',
    'page_type': 'product_detail',
    'category': 'Electronics',
    'seed_skus': ['HPH-AUDIO-1000'],
    'candidate_skus': ['HPH-AUDIO-PRO', 'HPH-TRAVEL-CASE', 'HPH-USB-C-HUB'],
    'candidates': [
        {
            'sku': 'HPH-AUDIO-PRO',
            'score': 0.72,
            'reason_codes': ['explicit_candidate', 'premium_audio'],
            'evidence': [
                {'source': 'explicit', 'reason': 'Premium audio candidate for the current intent', 'weight': 0.05, 'attributes': {'category': 'Electronics', 'in_stock': True}},
                {'source': 'product_correlation', 'reason': 'Correlated with travel audio browsing context', 'weight': 0.11, 'source_sku': 'HPH-AUDIO-1000', 'attributes': {'category': 'Electronics', 'in_stock': True}},
            ],
        },
        {
            'sku': 'HPH-TRAVEL-CASE',
            'score': 0.64,
            'reason_codes': ['bundle_product'],
            'evidence': [
                {'source': 'bundle', 'reason': 'Travel case pairs with headphones', 'weight': 0.10, 'source_sku': 'HPH-AUDIO-1000', 'attributes': {'category': 'Accessories', 'in_stock': True}},
            ],
        },
        {
            'sku': 'HPH-BUDGET-EARBUDS',
            'score': 0.58,
            'reason_codes': ['substitute_product'],
            'evidence': [
                {'source': 'substitute', 'reason': 'Lower-price audio substitute', 'weight': 0.03, 'source_sku': 'HPH-AUDIO-1000', 'attributes': {'category': 'Electronics', 'in_stock': True}},
            ],
        },
    ],
    'cart_skus': ['HPH-USB-C-HUB'],
    'max_candidates': 8,
    'context': {
        'loyalty_tier': 'gold',
        'channel': 'web',
        'region': 'US',
        'presentation': 'Retailer IQ demo',
    },
}

display(Markdown('### Recommendation-agent Sample Input'))
display(HTML('<p style="color:#475569">This payload is intentionally service-shaped: customer/session refs are scoped, candidate evidence is explicit, and cart exclusions are represented as context for the ranker.</p>'))
display(HTML(f'<pre style="white-space:pre-wrap; background:#0f172a; color:#f8fafc; padding:16px; border-radius:12px;">{html.escape(json.dumps(CANDIDATE_REQUEST, indent=2))}</pre>'))

In [ ]:
def _tool_attempts(tool_path: str, payload: dict[str, Any]) -> list[tuple[str, str, str, dict[str, Any] | None]]:
    tool_name = tool_path.lstrip('/')
    encoded_tool = parse.quote(tool_name, safe='')
    attempts: list[tuple[str, str, str, dict[str, Any] | None]] = [
        ('mcp-path', 'POST', f'/agents/search-enrichment-agent/mcp/{tool_name}', payload),
    ]
    if encoded_tool != tool_name:
        attempts.append(('mcp-encoded', 'POST', f'/agents/search-enrichment-agent/mcp/{encoded_tool}', payload))
    direct_method = 'GET' if tool_path == '/models/status' else 'POST'
    attempts.append(('agent-direct-route', direct_method, f'/agents/search-enrichment-agent{tool_path}', None if direct_method == 'GET' else payload))
    return attempts

def call_recommendation_tool(tool_path: str, payload: dict[str, Any]) -> dict[str, Any]:
    attempts: list[HttpCallResult] = []
    for label, method, path, request_payload in _tool_attempts(tool_path, payload):
        result = call_http(method, path, payload=request_payload, label=f'{label}:{tool_path}')
        attempts.append(result)
        if result.ok:
            return {'ok': True, 'tool': tool_path, 'route': label, 'result': result, 'attempts': attempts}
    return {'ok': False, 'tool': tool_path, 'route': None, 'result': attempts[-1], 'attempts': attempts}

def offline_candidates(payload: dict[str, Any]) -> list[dict[str, Any]]:
    merged: dict[str, dict[str, Any]] = {}
    for candidate in payload.get('candidates', []):
        merged[candidate['sku']] = dict(candidate)
    for sku in payload.get('candidate_skus', []):
        merged.setdefault(sku, {
            'sku': sku,
            'score': 0.55,
            'reason_codes': ['explicit_candidate'],
            'evidence': [{'source': 'explicit', 'reason': 'Candidate supplied by caller', 'weight': 0.0, 'attributes': {'in_stock': True}}],
        })
    return sorted(merged.values(), key=lambda item: (-float(item.get('score', 0.0)), item['sku']))[: payload.get('max_candidates', 8)]

def offline_rank(candidates: list[dict[str, Any]], cart_skus: list[str]) -> list[dict[str, Any]]:
    ranked: list[dict[str, Any]] = []
    for candidate in candidates:
        score = float(candidate.get('score', 0.5))
        reason_codes = list(candidate.get('reason_codes', []))
        for evidence_item in candidate.get('evidence', []):
            score += float(evidence_item.get('weight', 0.0))
            if evidence_item.get('source') == 'bundle' and 'bundle_product' not in reason_codes:
                reason_codes.append('bundle_product')
            if evidence_item.get('source') == 'product_correlation' and 'product_correlation' not in reason_codes:
                reason_codes.append('product_correlation')
        if candidate['sku'] in cart_skus:
            score -= 0.4
            reason_codes.append('already_in_cart')
        ranked.append({
            'sku': candidate['sku'],
            'score': round(max(0.0, min(1.0, score)), 4),
            'reason_codes': list(dict.fromkeys(reason_codes or ['baseline_score'])),
            'evidence': candidate.get('evidence', []),
        })
    ranked = sorted(ranked, key=lambda item: (-item['score'], item['sku']))
    for index, item in enumerate(ranked, start=1):
        item['rank'] = index
    return ranked

def ranked_to_candidates(ranked: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [
        {'sku': item['sku'], 'score': item['score'], 'reason_codes': item.get('reason_codes', []), 'evidence': item.get('evidence', [])}
        for item in ranked
    ]

def render_pipeline(pipeline: dict[str, dict[str, Any]], ranked: list[dict[str, Any]], cards: list[dict[str, Any]]) -> None:
    step_cards = []
    for name, call in pipeline.items():
        result: HttpCallResult = call['result']
        status_key, status_label = classify_call(result)
        mode = 'live' if call['ok'] else 'fallback-ready'
        code = 'n/a' if result.status_code is None else str(result.status_code)
        step_cards.append(
            f"<div class='pipe-card {status_key}'><div class='pipe-name'>{html.escape(name)}</div>"
            f"<div class='pipe-mode'>{html.escape(mode)} via {html.escape(str(call.get('route') or 'no live route'))}</div>"
            f"<div class='pipe-meta'>HTTP {code} | {round(result.latency_ms, 1)} ms</div></div>"
        )
    ranked_rows = ''.join(
        f"<tr><td>{item.get('rank')}</td><td><code>{html.escape(item['sku'])}</code></td><td>{item['score']:.3f}</td><td>{html.escape(', '.join(item.get('reason_codes', [])))}</td></tr>"
        for item in ranked[:6]
    )
    card_html = ''.join(
        f"<div class='rec-card'><strong>{html.escape(card['sku'])}</strong><span>{card['score']:.3f}</span><p>{html.escape(', '.join(card.get('reason_codes', [])))}</p></div>"
        for card in cards[:3]
    )
    display(HTML(f'''
<style>
.pipeline-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap: 10px; margin: 14px 0; }}
.pipe-card {{ border-radius: 12px; padding: 12px; border: 1px solid #e2e8f0; background: #f8fafc; }}
.pipe-card.ready {{ background: #ecfdf5; border-color: #34d399; }}
.pipe-card.auth {{ background: #eff6ff; border-color: #93c5fd; }}
.pipe-card.backend {{ background: #fff7ed; border-color: #fdba74; }}
.pipe-card.route {{ background: #fef2f2; border-color: #fca5a5; }}
.pipe-name {{ font-weight: 950; color: #111827; }}
.pipe-mode {{ color: #0f766e; font-size: 12px; margin-top: 6px; font-weight: 800; }}
.pipe-meta {{ color: #64748b; font-size: 12px; margin-top: 6px; }}
.rank-table {{ width: 100%; border-collapse: collapse; margin-top: 12px; }}
.rank-table th {{ text-align: left; background: #164e63; color: #fff; padding: 8px; }}
.rank-table td {{ border-bottom: 1px solid #e5e7eb; padding: 8px; }}
.rec-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 10px; margin-top: 12px; }}
.rec-card {{ border: 1px solid #d1fae5; border-radius: 12px; padding: 12px; background: #f0fdfa; }}
.rec-card strong {{ display: block; color: #0f172a; }}
.rec-card span {{ display: inline-block; margin-top: 6px; color: #047857; font-weight: 900; }}
.rec-card p {{ margin: 6px 0 0; color: #475569; font-size: 12px; }}
</style>
<h2>Live Recommendation-agent Pipeline</h2>
<p style='color:#475569'>The notebook tries APIM MCP first, then encoded MCP, then direct APIM agent routes. If the live path is unavailable, the local preview uses the same deterministic baseline shape for presentation continuity.</p>
<div class='pipeline-grid'>{''.join(step_cards)}</div>
<h3>Ranked Output</h3>
<table class='rank-table'><thead><tr><th>Rank</th><th>SKU</th><th>Score</th><th>Reason codes</th></tr></thead><tbody>{ranked_rows}</tbody></table>
<h3>UI Card Preview</h3>
<div class='rec-grid'>{card_html}</div>
'''))

pipeline_calls: dict[str, dict[str, Any]] = {}
pipeline_calls['model_status'] = call_recommendation_tool('/models/status', {})
pipeline_calls['candidates'] = call_recommendation_tool('/recommendations/candidates', CANDIDATE_REQUEST)

candidate_body = pipeline_calls['candidates']['result'].body if pipeline_calls['candidates']['ok'] else {}
candidate_rows = candidate_body.get('candidates') if isinstance(candidate_body, dict) else None
candidate_rows = candidate_rows or offline_candidates(CANDIDATE_REQUEST)

rank_request = {
    'tenant_id': CANDIDATE_REQUEST['tenant_id'],
    'customer_ref': CANDIDATE_REQUEST['customer_ref'],
    'session_id': CANDIDATE_REQUEST['session_id'],
    'intent': CANDIDATE_REQUEST['intent'],
    'query': CANDIDATE_REQUEST['query'],
    'category': CANDIDATE_REQUEST['category'],
    'cart_skus': CANDIDATE_REQUEST['cart_skus'],
    'candidates': candidate_rows,
    'max_items': 5,
    'context': CANDIDATE_REQUEST['context'],
}
pipeline_calls['rank'] = call_recommendation_tool('/recommendations/rank', rank_request)
rank_body = pipeline_calls['rank']['result'].body if pipeline_calls['rank']['ok'] else {}
ranked_rows = rank_body.get('ranked') if isinstance(rank_body, dict) else None
ranked_rows = ranked_rows or offline_rank(candidate_rows, CANDIDATE_REQUEST['cart_skus'])

compose_request = {
    'tenant_id': CANDIDATE_REQUEST['tenant_id'],
    'customer_ref': CANDIDATE_REQUEST['customer_ref'],
    'session_id': CANDIDATE_REQUEST['session_id'],
    'ranked_items': ranked_to_candidates(ranked_rows),
    'max_items': 3,
    'context': CANDIDATE_REQUEST['context'],
}
pipeline_calls['compose'] = call_recommendation_tool('/recommendations/compose', compose_request)
compose_body = pipeline_calls['compose']['result'].body if pipeline_calls['compose']['ok'] else {}
card_rows = compose_body.get('cards') if isinstance(compose_body, dict) else None
card_rows = card_rows or ranked_to_candidates(ranked_rows[:3])

top_item = ranked_rows[0]
explain_request = {
    'tenant_id': CANDIDATE_REQUEST['tenant_id'],
    'sku': top_item['sku'],
    'customer_ref': CANDIDATE_REQUEST['customer_ref'],
    'session_id': CANDIDATE_REQUEST['session_id'],
    'reason_codes': top_item.get('reason_codes', []),
    'evidence': top_item.get('evidence', []),
    'natural_language': False,
    'context': CANDIDATE_REQUEST['context'],
}
pipeline_calls['explain'] = call_recommendation_tool('/recommendations/explain', explain_request)
feedback_request = {
    'tenant_id': CANDIDATE_REQUEST['tenant_id'],
    'recommendation_set_id': (compose_body.get('recommendation_set_id') if isinstance(compose_body, dict) else None) or 'demo-offline-recset',
    'event_type': 'impression',
    'sku': top_item['sku'],
    'customer_ref': CANDIDATE_REQUEST['customer_ref'],
    'session_id': CANDIDATE_REQUEST['session_id'],
    'metadata': {'source': 'retailer-iq-notebook'},
}
pipeline_calls['feedback'] = call_recommendation_tool('/recommendations/feedback', feedback_request)

render_pipeline(pipeline_calls, ranked_rows, card_rows)

In [ ]:
def _pipeline_ok(step: str) -> bool:
    return bool(globals().get('pipeline_calls', {}).get(step, {}).get('ok'))

def _health_ok(service_name: str) -> bool:
    for row in globals().get('health_results', []):
        if row.get('service') == service_name:
            return bool(row.get('ok'))
    return False

def render_readiness_gates() -> None:
    live_compose_body = pipeline_calls.get('compose', {}).get('result').body if globals().get('pipeline_calls') else {}
    ready_for_ui = bool(isinstance(live_compose_body, dict) and live_compose_body.get('ready_for_ui'))
    gates = [
        ('Service route', _health_ok('search-enrichment-agent'), 'search-enrichment-agent /health is reachable through APIM'),
        ('Model status', _pipeline_ok('model_status'), '/models/status returns active model governance metadata'),
        ('Candidates', bool(candidate_rows), 'Candidate set is non-empty and evidence-backed'),
        ('Rank', bool(ranked_rows), 'Ranked list exists with bounded scores and reason codes'),
        ('Compose', bool(card_rows), 'Composable recommendation cards exist'),
        ('Feedback', _pipeline_ok('feedback'), 'Feedback endpoint accepted or live path is visible'),
        ('UI readiness', ready_for_ui or bool(card_rows), 'Cards are ready to render or notebook fallback preview is available'),
    ]
    items = []
    for name, ok, detail in gates:
        color = '#047857' if ok else '#c2410c'
        bg = '#ecfdf5' if ok else '#fff7ed'
        label = 'pass' if ok else 'attention'
        items.append(
            f"<div class='gate' style='background:{bg}'><span style='color:{color}'>{label}</span><strong>{html.escape(name)}</strong><p>{html.escape(detail)}</p></div>"
        )
    score = sum(1 for _, ok, _ in gates if ok)
    display(HTML(f'''
<style>
.gate-wrap {{ margin: 18px 0; }}
.gate-score {{ border-radius: 16px; padding: 18px; color: #ffffff; background: linear-gradient(135deg, #0f766e, #164e63); margin-bottom: 14px; }}
.gate-score strong {{ font-size: 38px; display: block; }}
.gate-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(230px, 1fr)); gap: 10px; }}
.gate {{ border-radius: 12px; border: 1px solid #e2e8f0; padding: 12px; }}
.gate span {{ text-transform: uppercase; font-size: 11px; font-weight: 950; }}
.gate strong {{ display: block; color: #111827; margin-top: 5px; }}
.gate p {{ color: #475569; margin: 6px 0 0; font-size: 12px; line-height: 1.45; }}
</style>
<section class='gate-wrap'>
  <div class='gate-score'><strong>{score}/{len(gates)}</strong>Browser-readiness gates passing for this run</div>
  <div class='gate-grid'>{''.join(items)}</div>
</section>
'''))

render_readiness_gates()

## Presenter Close

The recommendation system is larger than one agent: it is a Retailer IQ capability that composes product truth, customer context, intent, inventory, delivery promise, policy, and feedback. The <code>recommendation-agent</code> is the manager of that capability, hosted today by <code>search-enrichment-agent</code>, with a deterministic baseline that can later be swapped for a tracked classical ML model.

For the next evolution, wire model metadata to a registry, publish recommendation REST routes through APIM if direct browser-facing routes are desired, and promote only neutral cross-service contracts into <code>holiday_peak_lib</code> after a second independent service needs them.